# Universidad Autónoma de Aguascalientes

# Departamento: Ciencias de la Computación

# Carrera: Ingeniería en Computación Inteligente

## Curso: Aprendizaje Automático

## Maestro: Dr. Francisco Javier Luna Rosas

## Alumno: Carlos Leonardo Cruz Ortiz

### Semestre: Enero-Junio del 2026

# PRACTICA 9. REDES NEURONALES (SCIKIT-LEARN_MLPCLASSIFIER) PARA IMPLEMENTAR LAS SIGUIENTES FORMAS DE ENTRENAMIENTO: TABLA-TESTING, TABLA-COMPLETA, LOOCV Y K-FOLDS. UTILICE EL DATASET DE CRITICAS DE CINE.
### En esta práctica se aplica los cuatro métodos de evaluación (Tabla Testing, Tabla Completa, LOOCV y K-Fold Cross Validation) a un modelo MLPClassifier para clasificar sentimientos en críticas de cine. Se utiliza embeddings Word2Vec para convertir las críticas de texto en vectores numéricos que el modelo pueda procesar. Se efectuan 10 iteraciones de cada método y se comparara los resultados.

## Paso 1: Importar librerías necesarias

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from gensim.models import Word2Vec
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split, LeaveOneOut, KFold
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score
import warnings
warnings.filterwarnings('ignore')

## Paso 2: Cargamos el dataset de críticas de cine y datos auxiliares para procesamiento de texto

In [ ]:
movie_reviews = pd.read_csv("../data/movie_data.csv")
stop_words = set(stopwords.words("english"))

movie_reviews['review'] = movie_reviews['review'].str.lower()
movie_reviews = movie_reviews.dropna()

movie_reviews

In [ ]:
misspell_data = pd.read_csv("../data/aspell.txt", sep=":", names=["correction", "misspell"])
misspell_data["correction"] = misspell_data["correction"].str.lower()
misspell_data["misspell"] =  misspell_data["misspell"].str.strip()
misspell_data["misspell"] =  misspell_data["misspell"].str.split(" ")
misspell_data = misspell_data.explode('misspell').reset_index(drop=True)

# Creamos un diccionario para corregir las palabras mal escritas
miss2correction = dict(
  zip(misspell_data["misspell"], misspell_data["correction"])
)

In [ ]:
contractions_first = pd.read_csv("../data/contractions.csv", sep=",")

# Creamos un diccionario para expandir las contracciones
contr2meaning = dict(
  zip(contractions_first["contraction"], contractions_first["meaning"])
)

print(f"Contracciones del Dataset: {len(contr2meaning)}")

In [ ]:
contractions_second = None

with open("../data/contractions.json", "r", encoding="utf-8") as f:
  contractions_second = json.load(f)
  
print(f"Contracciones del Dataset: {len(contractions_second)}")

## Paso 3: Creamos la función de limpieza para las críticas de cine

In [ ]:
def clean_review(review: str) -> str:
  # Aglunas críticas tienen el tag "<br />", no queremos eso
  review = review.replace("<br />", " ").strip()
  words = review.split()
  
  for word in words:
    if word in miss2correction.keys():
      review = review.replace(word, miss2correction[word])
      
    if word in contr2meaning.keys():
      review = review.replace(word, contr2meaning[word])
      
  # Tokenizamos el texto y eliminamos las palabras vacías (stop words)
  # NOTA: solo separa las palabras, no crea tokens como los LLMs
  tokens = word_tokenize(review)
      
  return " ".join([
    word for word in tokens
    if word.isalpha() and word not in stop_words
  ])
  
# Testeamos la función de limpieza con una crítica de ejemplo
example_review = movie_reviews["review"].iloc[13]
print(f"Crítica original:\n{example_review[:300]}\n")
print(f"Crítica limpia:\n{clean_review(example_review)[:300]}")

## Paso 4: Limpieza de todas las críticas del dataset

In [ ]:
movie_reviews["clean_review"] = movie_reviews["review"].apply(clean_review)

In [ ]:
movie_reviews["clean_review"].head(20)

## Paso 5: Tokenización y entrenamiento del modelo Word2Vec

In [ ]:
tokenized_reviews = movie_reviews["clean_review"].str.split().tolist()
tokenized_reviews[0][:10]

In [ ]:
word2vec_model = Word2Vec(
  tokenized_reviews,
  vector_size=512,
  window=5,
  workers=4,
  min_count=3,
  sg=1
)

In [ ]:
import os

if not os.path.exists("../models/trainable_models/word2vec"):
  os.makedirs("../models/trainable_models/word2vec")  

word2vec_model.save("../models/trainable_models/word2vec/word2vec_formas_de_evaluacion.model")
word2vec_model.wv.save("../models/word_vectors_formas_de_evaluacion.wordvectors")

In [ ]:
def convert_to_vector(review: str) -> np.ndarray:
  return word2vec_model.wv.get_mean_vector(review.split(" "))

# Testeamos la función de conversión con una crítica de ejemplo
example_review = movie_reviews["clean_review"].iloc[13]
embedding_vector = convert_to_vector(example_review)
print(embedding_vector.shape)
print(embedding_vector[:10])

## Paso 6: Preparación de datos X e y para entrenamiento

In [ ]:
y = movie_reviews["sentiment"]
X = movie_reviews["clean_review"].apply(convert_to_vector)
X = pd.DataFrame(X.tolist())

y, X

## Paso 7: Método 1 - Tabla Testing (split 70/30)

In [ ]:
def evaluacion_tabla_testing(X, y, n_iteraciones=10):
  """
  Evalúa el modelo usando división train/test (70/30).

  :param X: Variables predictoras (vectores de críticas)
  :param y: Variable objetivo (sentimientos)
  :param int n_iteraciones: Número de iteraciones a ejecutar
  :return: list — Lista de errores por iteración
  """
  errores = []

  print("Ejecutando evaluación con Tabla Testing (70/30)...\n")

  for i in range(n_iteraciones):
    # Dividir datos en train y test
    X_train, X_test, y_train, y_test = train_test_split(
      X, y, train_size=0.7, random_state=i
    )

    # Crear y entrenar el modelo
    modelo = MLPClassifier(max_iter=200, random_state=i)
    modelo.fit(X_train, y_train)

    # Predecir y calcular error
    y_pred = modelo.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    error = 1 - accuracy
    errores.append(error)

    print(f"  Iteración {i+1}: Accuracy = {accuracy:.4f}, Error = {error:.4f}")

  print(f"\nError promedio (Tabla Testing): {np.mean(errores):.4f}")
  print(f"Desviación estándar: {np.std(errores):.4f}")
  return errores


errores_testing = evaluacion_tabla_testing(X, y, n_iteraciones=10)

## Paso 8: Método 2 - Tabla Completa

In [ ]:
def evaluacion_tabla_completa(X, y, n_iteraciones=10):
  """
  Evalúa el modelo entrenando y evaluando con todos los datos.

  :param X: Variables predictoras (vectores de críticas)
  :param y: Variable objetivo (sentimientos)
  :param int n_iteraciones: Número de iteraciones a ejecutar
  :return: list — Lista de errores por iteración
  """
  errores = []

  print("Ejecutando evaluación con Tabla Completa...\n")

  for i in range(n_iteraciones):
    # Crear y entrenar el modelo con todos los datos
    modelo = MLPClassifier(max_iter=200, random_state=i)
    modelo.fit(X, y)

    # Predecir con los mismos datos de entrenamiento
    y_pred = modelo.predict(X)
    accuracy = accuracy_score(y, y_pred)
    error = 1 - accuracy
    errores.append(error)

    print(f"  Iteración {i+1}: Accuracy = {accuracy:.4f}, Error = {error:.4f}")

  print(f"\nError promedio (Tabla Completa): {np.mean(errores):.4f}")
  print(f"Desviación estándar: {np.std(errores):.4f}")
  return errores


errores_completa = evaluacion_tabla_completa(X, y, n_iteraciones=10)

## Paso 9: Método 3 - LOOCV (Leave-One-Out Cross Validation)

In [ ]:
def evaluacion_loocv(X, y, n_iteraciones=10):
  """
  Evalúa el modelo usando Leave-One-Out Cross Validation.
  ADVERTENCIA: Muy costoso computacionalmente con datasets grandes.

  :param X: Variables predictoras (vectores de críticas)
  :param y: Variable objetivo (sentimientos)
  :param int n_iteraciones: Número de iteraciones a ejecutar
  :return: list — Lista de errores por iteración
  """
  errores = []
  loo = LeaveOneOut()

  print("Ejecutando evaluación con LOOCV...")
  print(f"NOTA: Se entrenarán {len(X)} modelos por iteración.")
  print(f"Total de entrenamientos: {len(X) * n_iteraciones}\n")

  for i in range(n_iteraciones):
    print(f"  Iteración {i+1}/{n_iteraciones} - Entrenando {len(X)} modelos...")
    predicciones_correctas = 0
    total_predicciones = 0

    # Para cada fold de LOOCV
    for train_index, test_index in loo.split(X):
      X_train, X_test = X[train_index], X[test_index]
      y_train, y_test = y[train_index], y[test_index]

      # Entrenar modelo
      modelo = MLPClassifier(max_iter=200, random_state=i)
      modelo.fit(X_train, y_train)

      # Predecir
      y_pred = modelo.predict(X_test)
      predicciones_correctas += (y_pred == y_test).sum()
      total_predicciones += len(y_test)

    accuracy = predicciones_correctas / total_predicciones
    error = 1 - accuracy
    errores.append(error)

    print(f"    Accuracy = {accuracy:.4f}, Error = {error:.4f}")

  print(f"\nError promedio (LOOCV): {np.mean(errores):.4f}")
  print(f"Desviación estándar: {np.std(errores):.4f}")
  return errores


errores_loocv = evaluacion_loocv(X, y, n_iteraciones=10)

## Paso 10: Método 4 - K-Fold Cross Validation (K-ésimo grupo)
### Validación cruzada dejando k datos fuera en cada iteración (usaremos k=5)

In [ ]:
def evaluacion_kfold(X, y, n_iteraciones=10, k=5):
  """
  Evalúa el modelo usando K-Fold Cross Validation.

  :param X: Variables predictoras (vectores de críticas)
  :param y: Variable objetivo (sentimientos)
  :param int n_iteraciones: Número de iteraciones a ejecutar
  :param int k: Número de folds
  :return: list — Lista de errores por iteración
  """
  errores = []

  print(f"Ejecutando evaluación con K-Fold (K={k})...\n")

  for i in range(n_iteraciones):
    kfold = KFold(n_splits=k, shuffle=True, random_state=i)
    predicciones_correctas = 0
    total_predicciones = 0

    # Para cada fold
    for train_index, test_index in kfold.split(X):
      X_train, X_test = X[train_index], X[test_index]
      y_train, y_test = y[train_index], y[test_index]

      # Entrenar modelo
      modelo = MLPClassifier(max_iter=200, random_state=i)
      modelo.fit(X_train, y_train)

      # Predecir
      y_pred = modelo.predict(X_test)
      predicciones_correctas += (y_pred == y_test).sum()
      total_predicciones += len(y_test)

    accuracy = predicciones_correctas / total_predicciones
    error = 1 - accuracy
    errores.append(error)

    print(f"  Iteración {i+1} (K={k}): Accuracy = {accuracy:.4f}, Error = {error:.4f}")

  print(f"\nError promedio (K-Fold, K={k}): {np.mean(errores):.4f}")
  print(f"Desviación estándar: {np.std(errores):.4f}")
  return errores


errores_kfold = evaluacion_kfold(X, y, n_iteraciones=10, k=5)

## Paso 11: Resumen de resultados

In [ ]:
# Crear DataFrame con los resultados
resultados = pd.DataFrame({
    'Iteración': range(1, 11),
    'Tabla Testing': errores_testing,
    'Tabla Completa': errores_completa,
    'LOOCV': errores_loocv,
    'K-Fold (K=5)': errores_kfold
})

print("\n" + "="*80)
print("TABLA DE RESULTADOS - ERRORES POR MÉTODO DE EVALUACIÓN")
print("="*80)
print(resultados.to_string(index=False))

print("\n" + "="*80)
print("ESTADÍSTICAS DESCRIPTIVAS")
print("="*80)
print(resultados.describe().loc[['mean', 'std', 'min', 'max']].to_string())

## Paso 12: Gráfica de Variación del Error
### Visualización comparativa de los errores en las 10 iteraciones

In [ ]:
# Configurar la gráfica
plt.figure(figsize=(12, 7))

# Crear las líneas para cada método
iteraciones = range(1, 11)

plt.plot(iteraciones, errores_testing, 'o-', color='brown', 
         label='Tabla Testing', linewidth=2, markersize=8)
plt.plot(iteraciones, errores_completa, 'o-', color='blue', 
         label='Tabla Completa', linewidth=2, markersize=8)
plt.plot(iteraciones, errores_loocv, 'o-', color='green', 
         label='Promedio uno afuera (LOOCV)', linewidth=2, markersize=8)
plt.plot(iteraciones, errores_kfold, 'o-', color='magenta', 
         label='K-ésimo grupo (K=5)', linewidth=2, markersize=8)

# Configurar títulos y etiquetas
plt.title('Variación del Error - Análisis de Sentimientos (Críticas de Cine)', 
          fontsize=16, fontweight='bold')
plt.xlabel('Iteración', fontsize=12)
plt.ylabel('Estimación del Error', fontsize=12)

# Configurar leyenda
plt.legend(loc='best', fontsize=11, frameon=True, shadow=True)

# Configurar la cuadrícula
plt.grid(True, alpha=0.3, linestyle='--')

# Configurar los ejes
plt.xlim(0.5, 10.5)
plt.xticks(iteraciones)

# Ajustar el layout
plt.tight_layout()

# Mostrar la gráfica
plt.show()

print("\nGráfica generada exitosamente.")

## Conclusiones

### En esta práctica se implemento y comparo las cuatro formas de evaluación para un modelo de clasificación con MLPClassifier. Pero ahora con las criticas de cine y utilizando Word2Vec para el procesamiento de texto. 

## Referencias

### SCIKIT-LEARN (Machine Learning in Python 2025). MLPClassifier. URL: https://scikit-learn.org/stable/modules/generated/sklearn.neural_network.MLPClassifier.html (Último acceso febrero 2025).
### SCIKIT-LEARN (Machine Learning in Python 2025). Cross-validation. URL: https://scikit-learn.org/stable/modules/cross_validation.html (Último acceso febrero 2025).